
### Step 1: Import API and libraries

In [60]:
import os
import sys
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr
import requests
import json

# Avoid Windows cp1252 console crashes when Gradio prints emoji for MCP startup logs.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")


### Step 2: Simple RAG with gaurd rails and Dynamic Context Injection

In [61]:
system_message= """You are a digital twin of zainab Ahmed sAFEER.  when people talk to you, you respond as zainab in first person: using her voice, personality and knowledge. Please use information ONLY between the *** . if you dont know an answer, dont make any assumptions, simply say you dont know. strictly use only this information we provided here. *** Here's information about zainab to help you embody hER: zAINAB is a software engineer in test with a master's degree in computer science engineering WORKING ON TRANSITIOING TO THE AI engineering track. 
She's based in Chicago USA And she has a master's DEGREE in Information Systems From Robert Morris University Illinois  AND Bachelor's degree in computer science engineering Bangalore .. 
She has a background in QA test automation using AI tools such as cursor and mCP And also manual testing in various backgrounds such as Healthcare, govtech And education. 
Other career history: SOFTWARE engineer intern AT Vit Bangalore Where SHE was awarded the best engineering project of the Year 2012  AND Where her team created an AUGMENTED Reality based application for the iPad. 
2015 she worked as a software engineer intern At a dental network at a healthcare Network IN CHICAGO wherein The inventory management SOFTWARE and the website was completely refactored. 
FROM 2022 to 2024 she worked for ID Tech camps- AN EDUCATIONAL COMPANY OFFERING stem COURSES FOR CHILDREN OF VARIOUS AGES. 
FROM 2024 to 2026 she worked at a company called Gworks Which created saas CLOUD Software for local governments where she was the lead tester for a high impact utility billing Hub ubhub for local governments Across America. 
This software was used by over 3 000 cities Across America. 
WHAT Drives her: the combination of problem solving, impact and continuous learning. 
I'm motivated by the idea that AI isn't just a technical field It's a way to redesign how people work the more friction from everyday tasks and create systems that think alongside us. 
I'm also driven by the pace of the field AI engineering requires curiosity adaptability and willingness to experiment and those are the strengths that I naturally lean on . 
The idea of constantly, learning , refining instructions improving models and designing reliable systems is exciting to me. 
Communication style: Friendly warm And Simplistic and kind AND engaging and inspiring and motivating. 
I am a member of women in Tech and love to inspire other women Join the  tech Revolution. 
fUN Facts about me: aCHIEVER OF A PERFECT SCORE AT THE GLOBAL IELTS SPOKEN (ENGLISH) EXAM. PASSED GOOGLE'S TECHNICAL INTERVIEW FOR A TEST ENGINEER ROLE. 
I'm ALSO a mother of four little ones: one of whom Sports an extra chromosome (DOWN SYNDROME). i AM AN ADVOACATE FOR KIDS  WITH DISABILITIES. 
I'm also a motivational public speaker- Inspiring women To live a life of purpose and FAITH. Spirituality and religion Are my other Hobbies.
 I'm in the process of writing a book For women I'm the theme of productivity Especially for motherS. *** """


#gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)


### Step 5: Dynamic Context Injection

In [62]:
Topic_Context={
    "2001":"*** in 2001, Zainab was in 6th grade and was passionate about science and technology. She participated in various science fairs and won several awards for her innovative projects. Her love for technology was evident from a young age, and she often spent hours tinkering with gadgets and learning how they worked. ***",
    "cooking":"*** Zainab is a foodie and loves to cook. She enjoys experimenting with different cuisines and flavors in the kitchen. Cooking is not just a hobby for her, but also a way to unwind and express her creativity. She often shares her culinary adventures on social media, inspiring others to try new recipes and explore the world of food. ***",
    "travel":"*** Zainab is an avid traveler and loves to explore new places. She believes that traveling broadens the mind and allows her to experience different cultures and perspectives. Whether it's a weekend getaway or an international trip, Zainab is always up for an adventure and enjoys immersing herself in new environments. ***",
    "pizza":"*** Zainab has a particular fondness for pizza. She loves trying different toppings and styles of pizza from around the world. Whether it's a classic Margherita or a more adventurous combination, pizza is one of her go-to comfort foods. She often jokes that if she could eat pizza every day, she would be the happiest person alive. ***"}

In [63]:


pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

# Create send_notification func

def send_notification(message: str):
    if not pushover_user or not pushover_token:
        return "Pushover credentials missing; skipped notification."

    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    response = requests.post(pushover_url, data=payload, timeout=15)
    response.raise_for_status()
    return "Notification sent successfully."

In [64]:
send_notification("Hello from the digital twin of Zainab! This is a test notification.")

<Response [200]>

In [65]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the phone of real world version of you via Pushover. Use this to alert the user about important events or updates or complete a task that the user has asked you to do. The message parameter should contain the content of the notification you want to send.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

In [66]:
tools = [{"type":"function", "function":send_notification_function}]

### Dice roll tool
Register dice_roll and append to 	ools (between **tools** and **handle_tool_call**).

In [ ]:
import random

# Simulates rolling a single six-sided die
def dice_roll():
    return random.randint(1, 6)

roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die or simulate a random number between 1 and 6.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

tools.append({"type": "function", "function": roll_dice_function})


In [67]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments or "{}")
        #print(f"Calling function {function_name}")  #For future debugging ;)

        #Route to the appropriate function based on function_name
        if function_name == "send_notification":
            send_notification(args["message"])
            content = f"Notification sent: {args['message']}"
        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"
        #elif function_name == "insert_function_name_3":
        #    content = insert_function_name_3(args["message"])
        #...
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }

        tool_results.append(tool_call_result)

    return tool_results

In [68]:
def respond_ai(message, history):
    # Inject dynamic context based on keywords in the message
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n" + context

    # Gradio ChatInterface history is list[dict(role, content)] in recent versions.
    # Keep only valid role/content entries for OpenAI messages.
    prior_messages = []
    for item in history or []:
        if isinstance(item, dict) and "role" in item and "content" in item:
            prior_messages.append({"role": item["role"], "content": item["content"]})

    messages = [
        {"role": "system", "content": system_message_enhanced},
        *prior_messages,
        {"role": "user", "content": message},
    ]

    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools,
    )

    assistant_message = response.choices[0].message

    # Check if model wants to call tools
    while assistant_message.tool_calls:
        tool_results = handle_tool_call(assistant_message.tool_calls)

        messages.append(
            {
                "role": "assistant",
                "content": assistant_message.content or "",
                "tool_calls": [tc.model_dump() for tc in assistant_message.tool_calls],
            }
        )
        messages.extend(tool_results)

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools,
        )
        assistant_message = response.choices[0].message

    return assistant_message.content or ""

In [ ]:
def respond_ai(message, history):
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n" + context

    prior_messages = []
    for item in history or []:
        if isinstance(item, dict) and "role" in item and "content" in item:
            prior_messages.append({"role": item["role"], "content": item["content"]})

    messages = [
        {"role": "system", "content": system_message_enhanced},
        *prior_messages,
        {"role": "user", "content": message},
    ]

    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools,
    )

    assistant_message = response.choices[0].message

    if assistant_message.tool_calls:
        tool_results = handle_tool_call(assistant_message.tool_calls)

        messages.append(
            {
                "role": "assistant",
                "content": assistant_message.content or "",
                "tool_calls": [tc.model_dump() for tc in assistant_message.tool_calls],
            }
        )
        messages.extend(tool_results)

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
        )
        assistant_message = response.choices[0].message

    return assistant_message.content or ""

In [ ]:
def respond_ai(message, history):
    # Inject dynamic context based on keywords in the message
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n" + context

    # Gradio ChatInterface history is list[dict(role, content)] in recent versions.
    prior_messages = []
    for item in history or []:
        if isinstance(item, dict) and "role" in item and "content" in item:
            prior_messages.append({"role": item["role"], "content": item["content"]})

    messages = [
        {"role": "system", "content": system_message_enhanced},
        *prior_messages,
        {"role": "user", "content": message},
    ]

    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools,
    )

    assistant_message = response.choices[0].message

    if assistant_message.tool_calls:
        tool_results = handle_tool_call(assistant_message.tool_calls)

        messages.append(
            {
                "role": "assistant",
                "content": assistant_message.content or "",
                "tool_calls": [tc.model_dump() for tc in assistant_message.tool_calls],
            }
        )
        messages.extend(tool_results)

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
        )
        assistant_message = response.choices[0].message

    return assistant_message.content or ""

In [69]:
gr.ChatInterface(fn=respond_ai).launch(
    inbrowser=False,
    server_name="127.0.0.1",
    server_port=7866,
    mcp_server=True,
)

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\Computer\Documents\AI_Engineering\ai_env\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\Computer\Documents\AI_Engineering\ai_env\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\Users\Computer\Documents\AI_Engineering\ai_env\Lib\site-packages\gradio\blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "c:\Users\Computer\Documents\AI_Engineering\ai_env\Lib\site-packages\gradio\blocks.py", line 1632, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  F